In [1]:
%pip install -e ..

Obtaining file:///home/jovyan/rl4greencrab
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for rl4greencrab (pyproject.toml) ... done
  Created wheel for rl4greencrab: filename=rl4greencrab-1.0.0-py2.py3-none-any.whl size=1121 sha256=b3794a774cccd371988b2f2b7c7b1d87d012620e7a3ca5dee4144c2b7417b3ac
  Stored in directory: /tmp/pip-ephem-wheel-cache-28uw4n2g/wheels/e9/e5/3a/c5b9d01b60d6e51f679f6b593621f0c93057929d3d78fb63fa
Successfully built rl4greencrab
  Attempting uninstall: rl4greencrab
    Found existing installation: rl4greencrab 1.0.0
    Uninstalling rl4greencrab-1.0.0:
      Successfully uninstalled rl4greencrab-1.0.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
from stable_baselines3 import PPO, TD3
from skopt import gp_minimize, gbrt_minimize 
from sb3_contrib import TQC, RecurrentPPO
from stable_baselines3.common.env_util import make_vec_env
from rl4greencrab import evaluate_agent, multiConstAction, simulator
import pandas as pd
import numpy as np
from rl4greencrab import plot_agent
import ray
from skopt import gp_minimize, gbrt_minimize 
from skopt.plots import plot_convergence, plot_objective
from rl4greencrab import TwoActNormalized, twoActEnv

In [3]:
def evaluateConstAct(x):
    config = {
        "w_mort_scale" : 600,
        "growth_k": 0.70,
        'random_start':True,
        'var_penalty_const': 0,
        'observation_type': 'count-biomass-time'
    }
    env = twoActEnv(config)
    agent = multiConstAction(env=env, action=np.array(x))
    m_reward = evaluate_agent(agent=agent, ray_remote=True).evaluate(n_eval_episodes=200)
    
    return - m_reward

In [ ]:
%%time
max_action = 3000
ray.init(
    num_cpus=30,                 
    include_dashboard=False,   
    ignore_reinit_error=True
)
res = gp_minimize(evaluateConstAct, 2*[(0.0, max_action)], n_calls = 150, verbose=True)
res.x
ray.shutdown()

In [5]:
ray.init(
    num_cpus=30,                 
    include_dashboard=False,   
    ignore_reinit_error=True
)
rew = evaluateConstAct(
    [0.0, 787.3281177947351]
)
print(rew)
ray.shutdown()

2026-03-10 07:02:32,203	INFO worker.py:2012 -- Started a local Ray instance.


False
{'CPU': 30.0, 'memory': 21851539047.0, 'object_store_memory': 9364945305.0, 'node:__internal_head__': 1.0, 'node:10.42.0.159': 1.0}
5.168879298521285
